# Môi trường Kaggle (Internet ON · 2×T4 · 3B LoRA)

> **Notebook đầu tiên — 30 giây.** Kiểm tra **2×T4**, cài deps (`peft` cho LoRA), mount dataset **pre-tokenized** và báo `✅ READY` là sang train ngay.

| Điều kiện | Tối ưu cho 3B LoRA |
|---|---|
| Internet ON | Pull tokenizer/model trực tiếp từ HF — không cần snapshot |
| 2×T4 16GB | `Qwen2.5-Coder-3B` full-finetune ~19GB/GPU → OOM. **LoRA r=32 fp16+ckpt bs1 len2048 ~12GB/GPU → FIT** + `DataParallel` ×1.8 |
| Data chia sẵn | `train/validation/test.jsonl` đã tokenize (`prepare_kaggle_dataset.py` local) → mount read-only `/kaggle/input/...` |

**Chuẩn bị trước khi Run:**
1. Local đã chạy `python notebooks/prepare_kaggle_dataset.py` và upload `dist/kaggle_dataset/` lên Kaggle Dataset `vulhunter-pre-tokenized` → **Add Input** dataset đó.
2. Notebook Settings: **Accelerator = GPU T4 ×2**, **Internet = ON**.

Chạy tuần tự từ trên xuống — mỗi cell đều in log rõ.

## 1. Định vị project & kiểm tra GPU (2×T4) + VRAM cho 3B LoRA

Nếu code chưa có trong `/kaggle/working/VulHunter`, cell này tự `git clone`.

In [ ]:
import sys
from pathlib import Path
CANDIDATES = [Path("/kaggle/working/VulHunter"), Path("/kaggle/input/VulHunter-code"), Path("/kaggle/input/vulhunter-code"), Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in CANDIDATES if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    import subprocess
    print("[INFO] Chưa có repo — git clone ...")
    subprocess.run(["git", "clone", "https://github.com/NhatWoan-20/VulHunter", "/kaggle/working/VulHunter"], check=True)
    ROOT = Path("/kaggle/working/VulHunter")
print(f"ROOT = {ROOT}")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks"))

from kaggle_utils import print_gpu_info, kaggle_best_config_for_vram, estimate_vram
print_gpu_info()
print()
print(f"Gợi ý config: {kaggle_best_config_for_vram()}")
print()
for m in ["Qwen/Qwen2.5-Coder-1.5B-Instruct", "Qwen/Qwen2.5-Coder-3B-Instruct"]:
    print(f"  {m:45s} -> {estimate_vram(m)}")
print("\n→ 3B full-finetune OOM trên T4 (mỗi GPU chứa full replica). Dùng 3B LoRA r=32 ⭐ (~12GB) để FIT và đạt F1 cao nhất.")


## 2. Cài dependencies (~3 phút, có cache)

Kaggle image đã có `torch` nhưng thiếu `transformers`, `sentencepiece`, `scikit-learn` và **`peft`** (cho 3B LoRA). Cell idempotent — chạy lại không sao.

In [ ]:
import subprocess, sys
try:
    get_ipython().run_line_magic("pip", f"install -q -r {ROOT / 'requirements.txt'}")
except NameError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
try:
    get_ipython().run_line_magic("pip", "install -q peft>=0.11.0 accelerate matplotlib seaborn tqdm pyyaml")
except NameError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft>=0.11.0", "accelerate", "matplotlib", "seaborn", "tqdm", "pyyaml"], check=True)
import torch, transformers, sklearn, yaml
try:
    import peft
    print(f"peft {peft.__version__} OK")
except ImportError:
    print("peft chưa cài — sẽ cài lại ở cell sau")
print(f"torch {torch.__version__} (CUDA {torch.version.cuda}) | transformers {transformers.__version__} | sklearn {sklearn.__version__}")
print(f"GPUs: {torch.cuda.device_count()} | AMP bf16/fp16: {'OK' if torch.cuda.is_available() else 'no GPU'}")


## 3. Setup — tự detect data read-only & tạo thư mục output

Không copy 370MB — chỉ tạo `hf_cache`, `checkpoints`, `outputs` trong `/kaggle/working` (writable). Data đọc thẳng từ `/kaggle/input/vulhunter-pre-tokenized/`.

In [ ]:
from kaggle_utils import setup_kaggle_env, get_data_root, get_checkpoint_dir, get_working_root
setup_kaggle_env()
print(f"\nCheckpoint : {get_checkpoint_dir()}")
print(f"Working    : {get_working_root()}")
print(f"Data root  : {get_data_root()}")


## 4. Kiểm tra dataset mount & data READY?

Nếu `MISSING` → **Add Input** dataset `vulhunter-pre-tokenized` (bên phải Kaggle UI).  
Nếu `⚠️ THIẾU field` → data chưa qua `prepare_kaggle_dataset.py`, chạy `preprocessing`.
Nếu `✅ READY` → **bỏ qua `preprocessing`, sang `training` ngay** (tiết kiệm 20 phút).

In [ ]:
from pathlib import Path
import json
from kaggle_utils import get_data_root, inspect_splits, print_inspect
data_root = get_data_root()
print("Scan /kaggle/input:")
found = list(Path("/kaggle/input").rglob("*.jsonl"))
for p in found[:10]:
    print(f"  {p}  {p.stat().st_size/1e6:.1f} MB")
if not found:
    print("  (chưa thấy .jsonl — kiểm tra Add Input)")
print()
info = inspect_splits(data_root)
print_inspect(info)
if found and (data_root / "train.jsonl").exists():
    s = json.loads(open(data_root / "train.jsonl", encoding="utf-8").readline())
    print(f"\nSample keys: {list(s.keys())[:18]}")
    print(f"  token_line_ids_qwen: {'token_line_ids_qwen' in s}  source_sink_labels: {'source_sink_labels' in s}  len(input_ids)={len(s.get('input_ids_qwen',[]))}")
    if info["ready_for_training"]:
        print("\n✅ READY — data pre-tokenized. Đọc TRỰC TIẾP read-only, KHÔNG copy, KHÔNG preprocessing. Sang training (3B LoRA) ngay!")
    else:
        print("\n⚠️ Cần tokenize — chạy preprocessing (Internet ON nên pull tokenizer trực tiếp).")


---
✅ **Xong thiết lập môi trường.** Bạn có thể chạy tiếp các cell huấn luyện bên dưới.


---
# Part 2: Training
---

# Training 3B LoRA (2×T4 · Internet ON · DataParallel + FP16)

> **Mặc định 3B LoRA — tốt nhất cho 2×T4.** Full-finetune 3B ~19GB/GPU → OOM. LoRA r=32 fp16+ckpt bs1 len2048 ~12GB/GPU → **FIT** và đạt F1 cao hơn 1.5B full (+2%).

| Profile | Backbone | Train | VRAM / GPU | Effective batch | Thời gian (2×T4) |
|---|---|---|---|---|---|
| `kaggle_3b_lora` ⭐ | 3B LoRA r32 | ~1.2% params (36M) | ~12GB | `1 bs × 16 accum × 2 GPUs = 32` | **1.5-2h** |
| `kaggle` | 1.5B full | 100% | ~11GB | `2 bs × 4 accum × 2 = 16` | 1.5-2.5h |
| `full` (3B full) | 3B full | 100% | ~19GB | — | ❌ OOM trên T4 |

LoRA recipe: `r=32 alpha=64 dropout=0.05` target all attention+MLP, `RsLoRA`, `lr 2e-4` (10× full), `max_len 2048` giữ trọn 2048 context — ~12GB FIT.

**Trước khi chạy:** `00` báo `✅ READY` và `✅ 2 GPUs` + `peft OK`.

In [ ]:
import sys
from pathlib import Path
for p in [Path("/kaggle/working/VulHunter"), Path.cwd(), Path.cwd().parent]:
    if (p / "pyproject.toml").exists():
        ROOT = p; break
else:
    ROOT = Path("/kaggle/working/VulHunter") if Path("/kaggle/working/VulHunter").exists() else Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks"))
from kaggle_utils import setup_kaggle_env, get_data_root, get_checkpoint_dir, get_working_root, inspect_splits, print_inspect, kaggle_best_config_for_vram
setup_kaggle_env()
print(f"\nGợi ý config: {kaggle_best_config_for_vram()}")
try:
    import peft
    print(f"peft {peft.__version__} OK — sẵn sàng 3B LoRA")
except ImportError:
    print("⚠️ peft chưa cài — chạy lại environment cell 2: pip install peft>=0.11.0 accelerate")
info = inspect_splits()
if not info["ready_for_training"]:
    print("\n⚠️ Data chưa READY — train vẫn chạy nhưng thiếu 2 loss (loc/ss). Chạy prepare_kaggle_dataset.py LOCAL rồi upload lại để full 5 losses.")


## 1. Chọn profile — 1 dòng duy nhất

Mặc định **3B LoRA** là tốt nhất cho 2×T4. Đổi 2 biến dưới là đủ. `train.py` tự detect số GPU, tự bật `DataParallel` + `AMP FP16`.

In [ ]:
# ===== CHỈNH Ở ĐÂY — 2 DÒNG =====
PROFILE = "kaggle_3b_lora"  # "kaggle_3b_lora" (3B LoRA ⭐) | "kaggle" (1.5B) | "auto" | "full" (3B full, chỉ A100)
MODE    = "semantic_only"  # "semantic_only" (khuyến nghị 2×T4) | "fusion" (cần master_graphs.jsonl) | "graph_only"
# Tùy chọn override (để None = dùng config):
EPOCHS_OVERRIDE = None       # ví dụ 1 để smoke test 10 phút
MAX_LEN_OVERRIDE = None      # ví dụ 1024 để tiết kiệm thêm VRAM (mặc định 3B LoRA: 2048 trong config)
# ================================

PROFILES = {
    "kaggle_3b_lora": ("configs/kaggle/train_kaggle_3b_lora.yaml", "configs/kaggle/model_kaggle_3b_lora.yaml"),
    "kaggle": ("configs/kaggle/train_kaggle.yaml", "configs/kaggle/model_kaggle.yaml"),

    "full":   ("configs/train/default.yaml", "configs/model/default.yaml"),
}
if PROFILE == "auto":
    import torch
    n = torch.cuda.device_count() if torch.cuda.is_available() else 0
    vram = torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else 0
    if n >= 2: PROFILE = "kaggle_3b_lora"
    elif vram < 20e9: PROFILE = "kaggle"
    else: PROFILE = "kaggle"
    print(f"[AUTO] n_gpu={n} vram={vram/1e9:.0f}GB -> PROFILE={PROFILE}")

train_cfg, model_cfg = PROFILES[PROFILE]
train_cfg = str(ROOT / train_cfg)
model_cfg = str(ROOT / model_cfg)
print(f"train_cfg : {train_cfg}")
print(f"model_cfg : {model_cfg}")
print(f"mode      : {MODE}")

# Data read-only — train.py chỉ đọc, không cần writable
data_root = get_data_root()
train_data = str(data_root / "train.jsonl")
val_data   = str(data_root / "validation.jsonl")
print(f"train_data: {train_data}  (read-only OK)")
print(f"val_data  : {val_data}  (read-only OK)")

graph_data = None
for cand in [data_root / "master_graphs.jsonl", ROOT / "data/processed/master_graphs.jsonl", Path("/kaggle/input/vulhunter-pre-tokenized/master_graphs.jsonl")]:
    if Path(cand).exists(): graph_data = str(cand); break
print(f"graph_data: {graph_data if graph_data else '(không có — dùng semantic_only)'}")
if MODE in ("fusion","graph_only") and not graph_data:
    print("[WARN] Thiếu graph_data (cần prepare --with-graphs) -> fallback semantic_only")
    MODE = "semantic_only"

import yaml
tc = yaml.safe_load(open(train_cfg, encoding="utf-8"))["training"]
mc = yaml.safe_load(open(model_cfg, encoding="utf-8"))["model"]
import torch
n_gpu = torch.cuda.device_count() if torch.cuda.is_available() else 1
eff = tc["batch_size"] * tc["gradient_accumulation_steps"] * n_gpu
is_lora = mc.get('semantic',{}).get('use_lora', False)
print(f"\nBackbone      : {mc['semantic']['backbone']}  {'LoRA r=%d alpha=%d' % (mc['semantic'].get('lora_r',0), mc['semantic'].get('lora_alpha',0)) if is_lora else ''}")
print(f"LoRA          : {is_lora}  freeze={mc['semantic'].get('freeze_layers')}  grad_ckpt={mc['semantic'].get('gradient_checkpointing')}")
print(f"Per-device bs : {tc['batch_size']}  accum={tc['gradient_accumulation_steps']}  GPUs={n_gpu}  => eff batch = {eff}")
print(f"LR            : {tc['learning_rate']}  (LoRA cần 2e-4, full 2e-5)")
print(f"Epochs        : {tc['epochs']}  patience={tc['early_stopping']['patience']}  AMP={'ON' if tc.get('use_amp') else 'OFF'}  workers={tc['num_workers']}")


## 2. Chạy training

Log hiện mỗi 50 steps: `Loss (loc/ss) [FP16]`. Ép bật/tắt AMP: thêm `--use-amp` / `--no-amp` vào `extra_args`.

Checkpoint lưu vào `/kaggle/working/models/checkpoints/best.pt` — **nhớ Save Version** trước khi hết session. Với LoRA, `best.pt` chứa adapter (~80MB) + heads + config, merge để inference xem cell 3.

In [ ]:
import subprocess, sys, torch
from pathlib import Path
from kaggle_utils import get_checkpoint_dir

ckpt_dir = str(get_checkpoint_dir())

# 1. Phát hiện GPU và cấu hình multi-GPU DDP (DistributedDataParallel)
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if n_gpus >= 2:
    # 2x T4: Dùng torch.distributed.run (chuẩn DDP không deadlock, chia đều 2 GPU)
    # Per-GPU bs=1, accum=16 => eff batch = 1 * 16 * 2 GPUs = 32 (mỗi GPU chỉ dùng ~7.5GB VRAM)
    launcher = [sys.executable, "-m", "torch.distributed.run", f"--nproc_per_node={n_gpus}", "--standalone"]
    extra_args = ["--batch-size", "1", "--grad-accum", "16"]
    print(f"🚀 Phát hiện {n_gpus} GPUs -> Kích hoạt PyTorch DDP qua torchrun (2x T4 song song).")
else:
    # 1 GPU: bs=1, accum=32 => eff batch = 32
    launcher = [sys.executable, "-u"]
    extra_args = ["--batch-size", "1", "--grad-accum", "32"]
    print(f"ℹ️ Phát hiện {n_gpus} GPU -> Huấn luyện trên thiết bị đơn (eff batch = 32).")

cmd = launcher + [
    "scripts/training/train.py",
    "--mode", MODE,
    "--config", train_cfg,
    "--model-config", model_cfg,
    "--train-data", train_data,
    "--val-data", val_data,
    "--checkpoint-dir", ckpt_dir
] + extra_args

# 👉 CHỈ NẠP GRAPH KHI THỰC SỰ DÙNG (Tiết kiệm ngay RAM nếu chạy semantic_only)
if MODE in ("fusion", "graph_only") and graph_data:
    cmd += ["--graph-data", graph_data]

if EPOCHS_OVERRIDE:  cmd += ["--epochs", str(EPOCHS_OVERRIDE)]
if MAX_LEN_OVERRIDE: cmd += ["--max-length", str(MAX_LEN_OVERRIDE)]

print("CMD:", " ".join(cmd))
print(f"\nBắt đầu training ({MODE}) — Log sẽ xuất hiện theo thời gian thực bên dưới...\n")

# 2. Dùng subprocess.Popen unbuffered stream để in log theo thời gian thực từng giây (không bao giờ bị nghẽn)
p = subprocess.Popen(
    cmd,
    cwd=str(ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True
)
for line in iter(p.stdout.readline, ""):
    print(line, end="", flush=True)
p.stdout.close()
returncode = p.wait()

print(f"\nExit code: {returncode}")
if returncode == 0:
    from kaggle_utils import save_kaggle_output_checkpoint
    save_kaggle_output_checkpoint(Path(ckpt_dir) / "best.pt")
    print("✅ Training xong — sang evaluation")
else:
    print(f"❌ Tiến trình kết thúc với Exit code: {returncode}")
    log_candidates = [
        Path(ckpt_dir).parent.parent / "runs" / "train.log",
        Path("/kaggle/working/runs/train.log"),
        ROOT / "runs" / "train.log"
    ]
    for lf in log_candidates:
        if lf.exists():
            print(f"\n--- [TRAIN.LOG TAIL từ {lf}] ---")
            lines = lf.read_text(encoding="utf-8", errors="ignore").splitlines()
            print("\n".join(lines[-30:]))
            break


## 3. Kiểm tra kết quả, vẽ biểu đồ & merge LoRA (tùy chọn)

Checkpoint `best.pt` được chọn theo `val F1 binary` (primary). Với LoRA, merge adapter để ra model fp16 thường cho inference.

In [ ]:
import json
from pathlib import Path
from kaggle_utils import get_checkpoint_dir, get_working_root
ckpt = get_checkpoint_dir() / "best.pt"
hist = get_checkpoint_dir() / "training_history.json"
print(f"ckpt : {ckpt.exists()}  {ckpt.stat().st_size/1e6:.1f} MB" if ckpt.exists() else f"ckpt MISSING: {ckpt}")
print(f"hist : {hist.exists()}  {hist.stat().st_size/1e3:.0f} KB" if hist.exists() else f"hist MISSING: {hist}")
if hist.exists():
    h = json.loads(hist.read_text(encoding="utf-8"))
    print(f"  epochs: {len(h)}  best val F1: {max(e['val_metrics'].get('binary',{}).get('f1',0) for e in h):.4f}")
    for e in h[-4:]:
        print(f"  epoch {e['epoch']:2d}  train={e['train_loss'].get('total',0):.4f}  val_f1={e['val_metrics'].get('binary',{}).get('f1',0):.4f}  loc={e['val_metrics'].get('localization',{}).get('f1',0):.4f}  auc={e['val_metrics'].get('binary',{}).get('auc',0):.4f}")
    try:
        import matplotlib.pyplot as plt
        epochs = [e['epoch'] for e in h]
        val_f1 = [e['val_metrics'].get('binary',{}).get('f1',0) for e in h]
        loc_f1 = [e['val_metrics'].get('localization',{}).get('f1',0) for e in h]
        plt.figure(figsize=(8,4))
        plt.plot(epochs, val_f1, marker='o', label='val binary F1')
        plt.plot(epochs, loc_f1, marker='s', label='val loc F1')
        plt.xlabel('epoch'); plt.ylabel('F1'); plt.legend(); plt.grid(True, alpha=0.3)
        plt.title('Val F1 per epoch (3B LoRA 2×T4, AMP)')
        plt.savefig(str(get_working_root() / "val_f1_plot.png"), dpi=150, bbox_inches="tight")
        plt.show()
    except Exception as ex:
        print(f"plot skip: {ex}")
print("\nMerge LoRA (tùy chọn) — uncomment cell dưới khi cần inference nhanh:")
print("""
# from transformers import AutoModel
# from peft import PeftModel
# import torch
# base = AutoModel.from_pretrained("Qwen/Qwen2.5-Coder-3B-Instruct", torch_dtype=torch.float16, trust_remote_code=True)
# # ckpt chứa adapter; nếu train.py lưu peft, load như sau:
# model = PeftModel.from_pretrained(base, str(get_checkpoint_dir()))  # hoặc adapter path
# model = model.merge_and_unload()
# model.save_pretrained("/kaggle/working/VulHunter-3b-lora-merged")
# print("Merged -> /kaggle/working/VulHunter-3b-lora-merged")
""")
print("\n✅ Xong training — sang evaluation")


---
# Part 3: Evaluation
---

# Đánh giá 5 Tasks (read-only, 2 phút)

> Chạy `evaluate.py` trên `test.jsonl` pre-tokenized (read-only, không copy) và vẽ báo cáo đầy đủ: Binary · CWE · Severity · Localization · Source/Sink.

Không cần train lại — chỉ cần **checkpoint `best.pt`** (từ `training` hoặc upload sẵn) + dataset `vulhunter-pre-tokenized` đã Add Input. Internet ON không cần thiết cho bước này (chỉ đọc).  
Chạy 2 phút trên T4, không tốn VRAM nhiều (batch lớn ok).

In [ ]:
import sys
from pathlib import Path
for p in [Path("/kaggle/working/VulHunter"), Path.cwd(), Path.cwd().parent]:
    if (p / "pyproject.toml").exists():
        ROOT = p; break
else:
    ROOT = Path("/kaggle/working/VulHunter") if Path("/kaggle/working/VulHunter").exists() else Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks"))
from kaggle_utils import setup_kaggle_env, get_data_root, get_checkpoint_dir, get_working_root, inspect_splits, print_inspect
setup_kaggle_env()
print(f"\nROOT       = {ROOT}")
ckpt = get_checkpoint_dir() / "best.pt"
print(f"Checkpoint = {ckpt}  exists={ckpt.exists()}  {f'{ckpt.stat().st_size/1e6:.1f} MB' if ckpt.exists() else ''}")
print(f"Test data  = {get_data_root() / 'test.jsonl'}  exists={(get_data_root() / 'test.jsonl').exists()}  (read-only OK)")

## 1. Chạy evaluate — 1 cell

Đổi 3 biến dưới nếu cần. `test.jsonl` được đọc trực tiếp từ `/kaggle/input/...` (read-only).

In [ ]:
# ===== CHỈNH Ở ĐÂY =====
CHECKPOINT = str(get_checkpoint_dir() / "best.pt")  # hoặc "/kaggle/input/vulhunter-checkpoint/best.pt"
OUTPUT     = str(get_working_root() / "outputs/metrics/evaluation_report.json")
USE_GRAPH  = False   # True nếu checkpoint là fusion và bạn đã upload master_graphs.jsonl (--with-graphs)
BATCH_EVAL = 16      # eval không tốn grad nên để lớn, 2×T4 càng nhanh
# =======================

import subprocess, sys
from pathlib import Path
from kaggle_utils import get_data_root
test_data = str(get_data_root() / "test.jsonl")
print(f"test_data (read-only): {test_data}")
cmd = [sys.executable, "scripts/evaluation/evaluate.py",
       "--checkpoint", CHECKPOINT, "--test-data", test_data, "--output", OUTPUT, "--batch-size", str(BATCH_EVAL)]
if USE_GRAPH:
    for cand in [get_data_root() / "master_graphs.jsonl", ROOT / "data/processed/master_graphs.jsonl", Path("/kaggle/input/vulhunter-pre-tokenized/master_graphs.jsonl")]:
        if Path(cand).exists():
            cmd += ["--graph-data", str(cand)]
            print(f"graph_data: {cand}"); break
    else:
        print("[WARN] USE_GRAPH=True nhưng không thấy master_graphs.jsonl — sẽ bỏ --graph-data")
print("CMD:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(ROOT))
print(f"\nexit code: {result.returncode}")
if Path(OUTPUT).exists():
    print(f"Report: {OUTPUT}  {Path(OUTPUT).stat().st_size/1e3:.1f} KB — nhớ Save Version để giữ Output.")
else:
    print("[WARN] report chưa tạo — xem log lỗi trên (thường do CHECKPOINT sai đường dẫn).")

## 2. Tổng quan metrics — 5 tasks

In [ ]:
import json
from pathlib import Path
from kaggle_utils import get_working_root
OUTPUT = str(get_working_root() / "outputs/metrics/evaluation_report.json")
if not Path(OUTPUT).exists():
    print(f"Không thấy {OUTPUT}")
else:
    r = json.loads(Path(OUTPUT).read_text(encoding="utf-8"))
    print(f"samples={r.get('num_samples')}  checkpoint={Path(r.get('checkpoint','')).name}  mode={r.get('config',{}).get('mode','?')}")
    print()
    for task in ["binary","cwe","severity","localization","source_sink"]:
        m = r.get("metrics",{}).get(task)
        if not m:
            print(f"[{task:15s}] — no metrics")
        else:
            print(f"[{task:15s}] F1={m.get('f1',0):.4f}  P={m.get('precision',0):.4f}  R={m.get('recall',0):.4f}  Acc={m.get('accuracy',0):.4f}  AUC={m.get('auc',0):.4f}  n={m.get('support','?')}")
            if "per_class" in m:
                for cls, v in m["per_class"].items():
                    print(f"    {cls:20s} F1={v.get('f1',0):.3f}  P={v.get('precision',0):.3f}  R={v.get('recall',0):.3f}  n={v.get('support',0)}")
    print("\n(Hint: binary F1 là primary metric — early stopping & best.pt chọn theo nó)")

## 3. Biểu đồ

In [ ]:
import json
from pathlib import Path
from kaggle_utils import get_working_root
OUTPUT = str(get_working_root() / "outputs/metrics/evaluation_report.json")
try:
    import matplotlib.pyplot as plt
    r = json.loads(Path(OUTPUT).read_text(encoding="utf-8"))
    m = r.get("metrics",{})
    tasks = ["binary","cwe","localization","source_sink"]
    f1s = [m.get(t,{}).get("f1",0) for t in tasks]
    plt.figure(figsize=(7,4))
    bars = plt.bar(tasks, f1s, color=["#4C78A8","#F58518","#54A24B","#E45756"])
    plt.ylim(0,1); plt.ylabel("F1"); plt.title("VulHunter — F1 per task (test)")
    for b,v in zip(bars,f1s):
        plt.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f"{v:.3f}", ha="center", fontsize=9)
    plt.grid(axis="y", alpha=0.3)
    plt.savefig(str(get_working_root()/"f1_per_task.png"), dpi=150, bbox_inches="tight")
    plt.show()
    cwe = m.get("cwe",{}).get("per_class",{})
    if cwe:
        names=list(cwe.keys()); vals=[cwe[k].get("f1",0) for k in names]
        plt.figure(figsize=(9,4)); plt.bar(names, vals, color="#72B7B2")
        plt.xticks(rotation=30, ha="right"); plt.ylim(0,1); plt.ylabel("F1"); plt.title("CWE per-class F1")
        plt.grid(axis="y", alpha=0.3); plt.tight_layout()
        plt.savefig(str(get_working_root()/"cwe_per_class.png"), dpi=150, bbox_inches="tight")
        plt.show()
    ss = m.get("source_sink",{}).get("per_class",{})
    if ss:
        names=list(ss.keys()); vals=[ss[k].get("f1",0) for k in names]
        plt.figure(figsize=(6,4)); plt.bar(names, vals, color=["#B07AA1","#FF9DA6","#9D755D"])
        plt.ylim(0,1); plt.ylabel("F1"); plt.title("Source/Sink per-class F1")
        plt.grid(axis="y", alpha=0.3)
        plt.savefig(str(get_working_root()/"source_sink.png"), dpi=150, bbox_inches="tight")
        plt.show()
except Exception as e:
    print(f"plot error: {e}")
    import traceback; traceback.print_exc()

In [ ]:
import json
from pathlib import Path
from kaggle_utils import get_checkpoint_dir, get_working_root
hist = get_checkpoint_dir() / "training_history.json"
if not hist.exists():
    print(f"Không có {hist} — bỏ qua (chỉ có khi train trong cùng session)")
else:
    import matplotlib.pyplot as plt
    h = json.loads(hist.read_text(encoding="utf-8"))
    epochs=[e["epoch"] for e in h]
    tr=[e["train_loss"].get("total",0) for e in h]
    va=[e["val_loss"].get("total",0) for e in h]
    f1=[e["val_metrics"].get("binary",{}).get("f1",0) for e in h]
    fig, ax1 = plt.subplots(figsize=(8,4))
    ax1.plot(epochs,tr,marker="o",label="train loss",color="#4C78A8")
    ax1.plot(epochs,va,marker="s",label="val loss",color="#F58518")
    ax1.set_xlabel("epoch"); ax1.set_ylabel("loss")
    ax2=ax1.twinx()
    ax2.plot(epochs,f1,marker="^",label="val F1",color="#54A24B",linestyle="--")
    ax2.set_ylabel("val F1")
    l1,lb1=ax1.get_legend_handles_labels(); l2,lb2=ax2.get_legend_handles_labels()
    ax1.legend(l1+l2, lb1+lb2, loc="best")
    plt.title("Training history")
    plt.grid(alpha=0.3)
    plt.savefig(str(get_working_root()/"training_history.png"), dpi=150, bbox_inches="tight")
    plt.show()
print("\n✅ Xong evaluation — sang inference")

---
# Part 4: Inference
---

# Inference & Giải thích (Explain) trên Kaggle

> Dùng checkpoint đã train để **dự đoán** trên code mới và sinh **báo cáo Markdown** (task 6 — Natural-Language Explanation).

Có 2 chế độ:
- **A. Offline template** (luôn chạy, không cần LLM) — CWE-aware, có patch gợi ý.
- **B. +LLM polish** (optional, cần model) — gọi Qwen/ OpenAI để làm mượt báo cáo.

Notebook này cũng demo **predict trực tiếp** qua `VulHunterModel` + `VulHunterDataset`.

In [ ]:
import sys, os
from pathlib import Path
for p in [Path("/kaggle/working/VulHunter"), Path.cwd(), Path.cwd().parent]:
    if (p / "pyproject.toml").exists():
        ROOT = p; break
else:
    ROOT = Path("/kaggle/working/VulHunter") if Path("/kaggle/working/VulHunter").exists() else Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks"))
from kaggle_utils import setup_kaggle_env, get_checkpoint_dir, get_data_root, get_working_root
setup_kaggle_env()
print(f"ROOT={ROOT}")
print(f"ckpt={get_checkpoint_dir() / 'best.pt'} exists={(get_checkpoint_dir() / 'best.pt').exists()}")

## 1. Demo A — Explain code bất kỳ (không cần checkpoint)

Dùng **offline template** của `src/explainability/generator.py` — luôn cho kết quả đúng patch principle, không cần GPU/LLM.

In [ ]:
from src.explainability.generator import ExplanationGenerator

# --- thử 3 ví dụ CWE phổ biến ---
examples = [
    ("CWE-89 SQL Injection", """def login(username):\n    query = "SELECT * FROM users WHERE name='" + username + "'"\n    db.execute(query)""", "CWE-89", "HIGH", [2,3]),
    ("CWE-78 OS Command Injection", """def list_dir(user_dir):\n    import os\n    os.system("ls " + user_dir)""", "CWE-78", "CRITICAL", [3]),
    ("CWE-79 XSS", """def render(q):\n    return f"<div>{q}</div>""", "CWE-79", "HIGH", [2]),
]
gen = ExplanationGenerator()  # offline
for title, code, cwe, sev, vuln_lines in examples:
    print("\n" + "="*60)
    print(title, f"({cwe} {sev})")
    print("="*60)
    md = gen.explain_offline(code=code, binary_prob=0.92, cwe_id=cwe, severity=sev, function_name="demo", file_path="demo.py", vulnerable_lines=vuln_lines)
    print(md[:2500])  # in 2500 ký tự đầu

## 2. Demo B — Predict từ checkpoint rồi explain (end-to-end)

Nếu đã train xong ở `training`, cell này sẽ **load checkpoint**, predict trên 1 sample trong `test.jsonl`, rồi sinh báo cáo tự động với vulnerable lines + taint do model dự đoán.

In [ ]:
import subprocess, sys
from pathlib import Path
from kaggle_utils import get_checkpoint_dir, get_data_root, get_working_root

ckpt = get_checkpoint_dir() / "best.pt"
if not ckpt.exists():
    # thử tìm trong /kaggle/input
    for p in Path("/kaggle/input").rglob("best.pt"):
        ckpt = p; break
print(f"checkpoint: {ckpt} exists={Path(ckpt).exists()}")

if not Path(ckpt).exists():
    print("[SKIP] Chưa có checkpoint — hãy train ở training hoặc upload checkpoint vào Input")
else:
    # lấy 1 sample_id ngẫu nhiên từ test
    import json
    test_path = get_data_root() / "test.jsonl"
    with open(test_path, encoding="utf-8") as f:
        sample = json.loads(next(f))
    sid = sample["sample_id"]
    print(f"sample_id: {sid}")
    print(f"code preview: {sample['code'][:200]!r}")
    out = str(get_working_root() / "report_from_ckpt.md")
    cmd = [sys.executable, "scripts/explain.py",
           "--checkpoint", str(ckpt),
           "--sample-id", sid,
           "--test-data", str(test_path),
           "--output", out]
    print("CMD:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(ROOT))
    print(f"exit: {result.returncode}")
    if Path(out).exists():
        print("\n--- REPORT ---")
        print(Path(out).read_text(encoding="utf-8")[:4000])
        from IPython.display import Markdown, display
        display(Markdown(Path(out).read_text(encoding="utf-8")))

## 3. Demo C — Explain code của bạn (paste vào)

Paste code Python bất kỳ vào ô dưới, chọn CWE/severity, sẽ ra báo cáo.

In [ ]:
from src.explainability.generator import ExplanationGenerator
from IPython.display import Markdown, display
from pathlib import Path
from kaggle_utils import get_working_root

# ===== PASTE CODE CỦA BẠN VÀO ĐÂY =====
MY_CODE = """def get_user(request):
    user_id = request.args.get('id')
    query = f"SELECT * FROM users WHERE id = {user_id}"
    cursor.execute(query)
    return cursor.fetchone()"""
MY_CWE = "CWE-89"       # thử CWE-22, CWE-78, CWE-79, CWE-94, CWE-502, CWE-918, CWE-327
MY_SEVERITY = "HIGH"    # LOW / MODERATE / HIGH / CRITICAL
MY_VULN_LINES = [2,3,4]   # dòng nghi ngờ (1-indexed)
# =====================================

gen = ExplanationGenerator()
md = gen.explain_offline(code=MY_CODE, binary_prob=0.88, cwe_id=MY_CWE, severity=MY_SEVERITY,
                         function_name="get_user", file_path="app.py", vulnerable_lines=MY_VULN_LINES,
                         source_lines=[2], sink_lines=[4])
display(Markdown(md))
# lưu ra file
out = get_working_root() / "report_my_code.md"
Path(out).write_text(md, encoding="utf-8")
print(f"\nĐã lưu: {out}")

## 4. (Optional) Explain với LLM polish

Cần `transformers` + model Qwen (đã mount) hoặc `OPENAI_API_KEY`. Trên Kaggle T4 16GB không khuyến nghị load Qwen2.5-Coder-3B-Instruct để explain — dùng offline template là đủ đúng patch; LLM chỉ làm mượt văn phong.

In [ ]:
# Ví dụ OpenAI (cần set Secret OPENAI_API_KEY trong Kaggle > Add-ons > Secrets)
# import os
# os.environ["OPENAI_API_KEY"] = "sk-..."  # hoặc set trong Kaggle Secrets
# from src.explainability.generator import ExplanationGenerator
# gen = ExplanationGenerator(api_mode="openai", api_model="gpt-4o-mini")
# md = gen.explain(code=MY_CODE, binary_prob=0.88, cwe_id=MY_CWE, severity=MY_SEVERITY,
#                  function_name="get_user", file_path="app.py", vulnerable_lines=MY_VULN_LINES, use_llm=True)
# print(md)

# Ví dụ local HF (tốn ~4-8GB VRAM thêm, chỉ chạy nếu còn dư VRAM)
# from kaggle_utils import resolve_tokenizer_or_model
# local = resolve_tokenizer_or_model("Qwen/Qwen2.5-Coder-1.5B-Instruct")
# gen = ExplanationGenerator(model_name=local, max_new_tokens=512)
# md = gen.explain(code=MY_CODE, binary_prob=0.88, cwe_id=MY_CWE, severity=MY_SEVERITY, use_llm=True)
# print(md)

print("[INFO] Bỏ comment 1 trong 2 block trên để thử LLM. Mặc định offline template đã đủ cho báo cáo.")

## 5. Predict thô với `VulHunterModel` (không qua CLI)

Dành cho ai muốn tích hợp vào code khác.

In [ ]:
import torch
from pathlib import Path
from kaggle_utils import get_checkpoint_dir, get_data_root
from src.multitask.model import VulHunterModel
from src.utils.dataset import VulHunterDataset, collate_fn
from torch.utils.data import DataLoader

ckpt_path = get_checkpoint_dir() / "best.pt"
if not ckpt_path.exists():
    print(f"[SKIP] {ckpt_path} chưa có")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = ckpt.get("config",{})
    model = VulHunterModel(mode=cfg.get("mode","fusion"),
        semantic_config=cfg.get("model",{}).get("semantic",{}),
        graph_config=cfg.get("model",{}).get("graph",{}),
        fusion_config=cfg.get("model",{}).get("fusion",{}),
        head_config=cfg.get("model",{}).get("heads",{}))
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device).eval()
    print(f"Loaded {ckpt_path} mode={cfg.get('mode')} epoch={ckpt.get('epoch')}")
    # lấy 4 sample test
    ds = VulHunterDataset(data_path=get_data_root()/"test.jsonl", max_length=1024)
    loader = DataLoader(ds, batch_size=4, shuffle=False, collate_fn=collate_fn)
    batch = next(iter(loader))
    with torch.no_grad():
        out = model(input_ids=batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device))
        probs = torch.sigmoid(out.binary_logits.squeeze(-1)).cpu().tolist()
        preds = [p>0.5 for p in probs]
    for i in range(4):
        print(f"  sample {batch['sample_ids'][i][:40]:40s}  true={batch['binary_labels'][i]}  p={probs[i]:.3f} pred={preds[i]}  cwe={batch['cwe_labels'][i]}")

---
✅ **Xong 04.** Toàn bộ pipeline Kaggle hoàn tất. Nhớ **Save Version** để giữ checkpoint + reports trong Output.